# Load Balancing, Hands-On

Companion notebook for **"System Design, Plainly — Part 1: Load Balancing"** (Nov 5, 2026).

Four small simulations, all runnable in seconds:

1. **The algorithm ladder** — random vs power-of-two-choices vs least-connections (balls into bins).
2. **Consistent hashing** — why `hash % N` reshuffles 80% of keys when you add a server, and why a ring moves only ~20%.
3. **Virtual nodes** — why one ring position per server is a lottery, and how 200 vnodes fix it.
4. **Cache-aware routing** — why a state-blind router re-pays the prefill bill on every LLM chat turn.


In [ ]:
import bisect, hashlib, math
import numpy as np

rng = np.random.default_rng(7)
print("numpy", np.__version__)


## 1. The ladder: random → power of two choices → least connections

Classic balls-into-bins. The theorem (Mitzenmacher, 1996/2001) is stated for **m = n** balls:
random assignment leaves the busiest bin at ~log n / log log n balls; power of two choices
(pick 2 bins at random, drop the ball in the emptier one) leaves it at log log n + O(1).
At n = 1,000,000 that's ~5.3 vs ~2.6 + O(1) — and the gap grows with n.


In [ ]:
def simulate(n, m, policy, rng):
    loads = np.zeros(n, dtype=int)
    for _ in range(m):
        if policy == "random":
            loads[rng.integers(n)] += 1
        elif policy == "p2c":
            a, b = rng.integers(n, size=2)
            loads[a if loads[a] <= loads[b] else b] += 1
        elif policy == "least-conn":
            loads[int(np.argmin(loads))] += 1
    return loads

# Theorem regime: m = n = 1,000 balls into 1,000 bins
n = 1000
for policy in ["random", "p2c", "least-conn"]:
    loads = simulate(n, n, policy, np.random.default_rng(7))
    print(f"m=n={n}: {policy:11s} max load = {loads.max()}  (mean = 1)")
print(f"theory: random ~ ln n / ln ln n = {math.log(n)/math.log(math.log(n)):.2f}; "
      f"P2C ~ log2(ln n) + O(1) = {math.log2(math.log(n)):.2f} + O(1)")

# Production regime: 10,000 requests, 100 servers
n, m = 100, 10_000
res = {}
for policy in ["random", "p2c", "least-conn"]:
    loads = simulate(n, m, policy, np.random.default_rng(7))
    res[policy] = loads.max()
    print(f"m={m}, n={n}: {policy:11s} busiest server = {loads.max()}  ({loads.max()/(m/n):.2f}x mean)")
assert res["p2c"] <= res["random"], "P2C must not be worse than random"
assert res["p2c"] <= 1.10 * (m / n), "P2C should sit within 10% of perfect balance"
assert res["least-conn"] == m // n, "least-connections with full knowledge is exactly balanced"
print("OK: one extra random pick buys almost all of perfect knowledge.")


## 2. Consistent hashing: adding a server

200,000 keys, 4 servers → 5 servers. Modulo placement (`hash(key) % N`) reshuffles nearly
everything; a consistent-hash ring moves only the new server's fair share (~20%).


In [ ]:
def h(b: bytes) -> int:
    return int(hashlib.md5(b).hexdigest(), 16)

K = 200_000
keys = [f"user:{i}".encode() for i in range(K)]

mod4 = [h(k) % 4 for k in keys]
mod5 = [h(k) % 5 for k in keys]
mod_moved = sum(a != b for a, b in zip(mod4, mod5)) / K
print(f"modulo: {mod_moved:.1%} of keys change servers")

def build_ring(n_nodes, vnodes=200):
    pts = sorted((h(f"n{node}#{v}".encode()), node)
                 for node in range(n_nodes) for v in range(vnodes))
    return pts

def owner_of(pts, key):
    kh = h(key)
    i = bisect.bisect_left(pts, (kh, -1))
    return pts[0 if i == len(pts) else i][1]

ring4, ring5 = build_ring(4), build_ring(5)
ring_moved = sum(owner_of(ring4, k) != owner_of(ring5, k) for k in keys) / K
print(f"ring (200 vnodes): {ring_moved:.1%} of keys change servers (ideal: 20.0%)")
assert mod_moved > 0.75, "modulo should move ~80%"
assert ring_moved < 0.25, "ring should move only ~the new node's share"
print("OK: the ring moves 4x fewer keys than modulo on a scale-out.")


## 3. Virtual nodes: one ring position per server is a lottery

With 10 servers and one token each, the largest arc of the ring is ~H(10)/10 ≈ 2.9× the fair
share in expectation — one unlucky server gets nearly triple the keys. Virtual nodes shrink
the spread like 1/√v.


In [ ]:
def ring_loads(n_nodes, vnodes):
    pts = build_ring(n_nodes, vnodes)
    counts = np.zeros(n_nodes, dtype=int)
    for k in keys:
        counts[owner_of(pts, k)] += 1
    return counts

for v in [1, 16, 64, 200]:
    c = ring_loads(10, v)
    print(f"{v:3d} vnodes/node: busiest = {c.max()/(K/10):.2f}x mean, quietest = {c.min()/(K/10):.2f}x mean")
c1, c200 = ring_loads(10, 1), ring_loads(10, 200)
assert c1.max() / (K / 10) > 1.8, "one token per node should be badly unbalanced"
assert c200.max() / (K / 10) < 1.25, "200 vnodes should be near-balanced"
print("OK: vnodes trade a bigger table for a fairer ring (Cassandra 4.0 defaults to 16 tokens + deliberate placement).")


## 4. Cache-aware routing: the LLM era

200 chat sessions, 8 turns each, 20 inference workers, a 4,000-token shared prefix per session
(system prompt + history). Round-robin sends each turn to a random worker, so the prefix is
almost never cached where the turn lands. Session affinity sends every turn to the same worker.


In [ ]:
W, S, TURNS, PREFIX = 20, 200, 8, 4000
rng2 = np.random.default_rng(7)
rr_hits = rr_tokens = sa_hits = sa_tokens = 0
for s in range(S):
    home = rng2.integers(W)
    for t in range(TURNS):
        if (s * TURNS + t) % W == home and t > 0:
            rr_hits += 1
        else:
            rr_tokens += PREFIX
        if t > 0:
            sa_hits += 1
        else:
            sa_tokens += PREFIX
followups = S * (TURNS - 1)
print(f"round-robin:      hit rate {rr_hits/followups:.1%}, prefill {rr_tokens/1e6:.2f}M tokens")
print(f"session affinity: hit rate {sa_hits/followups:.1%}, prefill {sa_tokens/1e6:.2f}M tokens")
print(f"affinity pays {rr_tokens/sa_tokens:.1f}x fewer prefill tokens")
assert abs(rr_hits / followups - 1 / W) < 0.02, "round-robin hits ~1/W of the time, as expected"
assert sa_hits == followups, "affinity caches every follow-up turn"
assert rr_tokens / sa_tokens > 5, "affinity should save >5x prefill in this toy"
print("OK: this is why SGLang's cache-aware router and the vLLM Router exist —")
print("   the load balancer is now part of the inference engine, not just the network.")
